# Phase 4E — frozen BIFOLD S2 validation baseline
Infrastructure only until every Phase 4D gate is satisfied. This notebook never opens sealed-test members.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_DIR = Path('/kaggle/working/SIH-26167-SATQuery')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', os.environ['SATQUERY_REPO_URL'], str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[multisensor]', 'configilm==0.7.0'], check=True)
REBEN_DIR = Path('/kaggle/working/reben-training-scripts')
if not REBEN_DIR.exists():
    subprocess.run(['git', 'clone', 'https://git.tu-berlin.de/rsim/reben-training-scripts.git', str(REBEN_DIR)], check=True)
subprocess.run(['git', 'checkout', '90f7a58a2757bb407df64dd01bfc62b79df2bdd5'], cwd=REBEN_DIR, check=True)
sys.path.insert(0, str(REBEN_DIR))


In [ ]:
import hashlib, json, shutil, tarfile
from pathlib import PurePosixPath
import zstandard
from ml.evaluation.phase4e_bifold_baseline import assert_phase4d_ready
EXPERIMENT_DIR = REPO_DIR / 'experiments/phase4_bigearthnet_multisensor'
READINESS = EXPERIMENT_DIR / 'phase4e_readiness.json'
assert_phase4d_ready(READINESS, EXPERIMENT_DIR / 'results/representative_raster_audit.json', EXPERIMENT_DIR / 'bifold_contract.json', EXPERIMENT_DIR / 'split_manifest.json')
readiness = json.loads(READINESS.read_text())
matches = list(Path('/kaggle/input').rglob('phase4_s2_selected.tar.zst'))
if len(matches) != 1:
    raise RuntimeError(f'Expected exactly one attached S2 package, found {len(matches)}')
package = matches[0]
digest = hashlib.sha256()
with package.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
if digest.hexdigest() != readiness['modalities']['s2']['package_sha256']:
    raise RuntimeError('Attached S2 package SHA-256 mismatch')
DATASET_ROOT = Path('/kaggle/working/phase4e-data')
with package.open('rb') as source, zstandard.ZstdDecompressor().stream_reader(source, read_across_frames=True) as stream, tarfile.open(fileobj=stream, mode='r|') as archive:
    for member in archive:
        path = PurePosixPath(member.name)
        if not member.isfile() or path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Unsafe packaged member: {member.name}')
        if path.parts[:2] != ('s2', 'validation'):
            continue
        target = DATASET_ROOT.joinpath(*path.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        extracted = archive.extractfile(member)
        if extracted is None:
            raise RuntimeError(f'Cannot extract packaged member: {member.name}')
        with target.open('wb') as output:
            shutil.copyfileobj(extracted, output)


In [ ]:
import torch
from reben_publication.BigEarthNetv2_0_ImageClassifier import BigEarthNetv2_0_ImageClassifier
from ml.evaluation.phase4e_bifold_baseline import BifoldS2Inference, Phase4DGatePaths, Phase4EProvenance, evaluate_validation_batches, iter_validation_batches, write_validation_artifacts
device = 'cuda' if torch.cuda.is_available() else 'cpu'
wrapper = BifoldS2Inference.from_pretrained(BigEarthNetv2_0_ImageClassifier, cache_dir=Path('/kaggle/working/hf-cache'), allow_network=True)
wrapper.model.to(device)
registration = wrapper.registration
provenance = Phase4EProvenance(experiment_name=os.environ['SATQUERY_EXPERIMENT_NAME'], git_sha=os.environ['SATQUERY_GIT_REF'], modality='s2', model_id=registration.model_id, model_revision=registration.revision, checkpoint_sha256=registration.checkpoint_sha256, preprocessing_profile=registration.preprocessing_profile, frozen_manifest_sha256=readiness['frozen_manifest_sha256'], materialized_package_sha256=readiness['modalities']['s2']['package_sha256'], threshold=0.5)
batches = iter_validation_batches(manifest_path=EXPERIMENT_DIR / 'split_manifest.json', dataset_root=DATASET_ROOT, profile=wrapper.profile, batch_size=64, device=device)
metrics, predictions = evaluate_validation_batches(batches, wrapper, provenance=provenance, gate_paths=Phase4DGatePaths(readiness=READINESS, raster_audit=EXPERIMENT_DIR / 'results/representative_raster_audit.json', preprocessing_contract=EXPERIMENT_DIR / 'bifold_contract.json', manifest=EXPERIMENT_DIR / 'split_manifest.json'))
output_dir = Path('/kaggle/working/satquery-output') / os.environ['SATQUERY_REMOTE_OUTPUT']
write_validation_artifacts(output_dir, provenance=provenance, metrics=metrics, predictions=predictions)
